In [1]:
import os
import csv
import numpy as np                  # For doing math
import matplotlib.pyplot as plt     # For plotting
import matplotlib.dates as mdates   # For formatting dates when plotting
import matplotlib.colors as colors  # For truncating colorbars
import matplotlib.style as style
import xarray as xr                 # For dealing with netCDF data
import pandas as pd                 # A quick way to deal with time stamps
import netCDF4 as nc                # Another way to deal with netCDF data
import glob
import metpy.calc as mpcalc
import datetime
import matplotlib.units as munits
import scipy.interpolate
import scipy.signal as sig
import scipy.stats as stats
import sys
import scipy.io as sio
engine = 'python'

In [2]:
# WS_E05_OBS = np.load('WS_E05_OBS.npy')
# WS_E06_OBS = np.load('WS_E06_OBS.npy')

# WD_E05_OBS = np.load('WD_E05_OBS.npy')
# WD_E06_OBS = np.load('WD_E06_OBS.npy')

TI_E05_OBS = np.load('TI_E05_OBS.npy')
TI_E06_OBS = np.load('TI_E06_OBS.npy')

--------------------BIAS------------------------

In [14]:
import os
import glob
import numpy as np
import pandas as pd

# Root where WS_all_runs is stored, change as necessary
# wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_all_runs"
wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_d02"

# Grab all WS_E05.npy and WS_E06.npy under each run directory
wrf_files = glob.glob(os.path.join(wrf_root, "*", "TI_E0*.npy"))

# Directory to save bias results (under the same parent as WS_all_runs)
bias_dir = os.path.join(os.path.dirname(wrf_root), "errors_bias_ti")
os.makedirs(bias_dir, exist_ok=True)

# Interpolate missing values in both E05 and E06 observational data
TI_E05_OBS_interpolated = (
    pd.DataFrame(TI_E05_OBS)
    .interpolate(method="linear", axis=0, limit_direction="both")
    .to_numpy()
)
TI_E06_OBS_interpolated = (
    pd.DataFrame(TI_E06_OBS)
    .interpolate(method="linear", axis=0, limit_direction="both")
    .to_numpy()
)

for f in sorted(wrf_files):
    ti_npy = np.load(f)

    # Decide which observation to use
    if "E05" in os.path.basename(f):
        obs_data = TI_E05_OBS_interpolated
        obs_label = "E05_OBS"
    elif "E06" in os.path.basename(f):
        obs_data = TI_E06_OBS_interpolated
        obs_label = "E06_OBS"
    else:
        print(f"Skipping file {f}: not matching E05 or E06.")
        continue

    # Quick sanity check on shapes
    if ti_npy.shape != obs_data.shape:
        print(f"Shape mismatch for {f}: WRF {ti_npy.shape}, OBS {obs_data.shape}")
        continue

    print(f"Processing {f} using observational data {obs_label}")

    # Bias array (same shape as obs/model: time × 10 heights)
    BIAS_TI = ti_npy - obs_data

    # Build a clean bias filename
    # e.g. WS_E05.npy → BIAS_WS_E05.npy
    base_name = os.path.basename(f)
    filename_bias = base_name.replace("TI_", "BIAS_TI_")

    # Optionally also include run name in the path
    run_name = os.path.basename(os.path.dirname(f))
    run_bias_dir = os.path.join(bias_dir, run_name)
    os.makedirs(run_bias_dir, exist_ok=True)

    output_path = os.path.join(run_bias_dir, filename_bias)
    np.save(output_path, BIAS_TI)

    print(f"Saved bias file: {output_path}")

Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54/TI_E05.npy using observational data E05_OBS
Saved bias file: /glade/work/mhafsah/Analysis_Proj_1/errors_bias_ti/era5_myj_vert54/BIAS_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54/TI_E06.npy using observational data E06_OBS
Saved bias file: /glade/work/mhafsah/Analysis_Proj_1/errors_bias_ti/era5_myj_vert54/BIAS_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80/TI_E05.npy using observational data E05_OBS
Saved bias file: /glade/work/mhafsah/Analysis_Proj_1/errors_bias_ti/era5_myj_vert80/BIAS_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80/TI_E06.npy using observational data E06_OBS
Saved bias file: /glade/work/mhafsah/Analysis_Proj_1/errors_bias_ti/era5_myj_vert80/BIAS_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert93/TI_E05.npy using observational data E05_OBS
Saved bias file: /glade/work/m

--------------------- centered RMSE --------------------------

In [16]:
import os
import glob
import numpy as np
import pandas as pd
import skill_metrics as sm  # centered_rms_dev

# Root where WS_all_runs is stored, change as it is necessary
# wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_all_runs"
wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_d02"

# Grab all WS_E05.npy and WS_E06.npy under each run directory
wrf_files = glob.glob(os.path.join(wrf_root, "*", "TI_E0*.npy"))

# Directory to save cRMSE results (sibling of WS_all_runs)
crmse_dir = os.path.join(os.path.dirname(wrf_root), "errors_crmse_ti")
os.makedirs(crmse_dir, exist_ok=True)

# Interpolate missing values in both E05 and E06 observational data
TI_E05_OBS_interpolated = (
    pd.DataFrame(TI_E05_OBS)
    .interpolate(method="linear", axis=0, limit_direction="both")
    .to_numpy()
)
TI_E06_OBS_interpolated = (
    pd.DataFrame(TI_E06_OBS)
    .interpolate(method="linear", axis=0, limit_direction="both")
    .to_numpy()
)

for f in sorted(wrf_files):
    ti_npy = np.load(f)

    # Decide which observation to use
    base = os.path.basename(f)
    if "E05" in base:
        obs_data = TI_E05_OBS_interpolated
        obs_label = "E05_OBS"
    elif "E06" in base:
        obs_data = TI_E06_OBS_interpolated
        obs_label = "E06_OBS"
    else:
        print(f"Skipping file {f}: not matching E05 or E06.")
        continue

    # Sanity check on shapes
    if ti_npy.shape != obs_data.shape:
        print(f"Shape mismatch for {f}: WRF {ti_npy.shape}, OBS {obs_data.shape}")
        continue

    print(f"Processing {f} using observational data {obs_label}")

    # cRMSE per height level → 10 numbers (not time-varying)
    cRMSE_TI = np.zeros(10)

    for i in range(10):
        # sm.centered_rms_dev(x, y) returns a scalar
        cRMSE_TI[i] = sm.centered_rms_dev(ti_npy[:, i], obs_data[:, i])

    # Build a clean CRMSE filename (parallel to bias block)
    # e.g. WS_E05.npy → CRMSE_WS_E05.npy
    filename_crmse = base.replace("TI_", "CRMSE_TI_")

    run_name = os.path.basename(os.path.dirname(f))
    run_crmse_dir = os.path.join(crmse_dir, run_name)
    os.makedirs(run_crmse_dir, exist_ok=True)

    output_path = os.path.join(run_crmse_dir, filename_crmse)
    np.save(output_path, cRMSE_TI)

    print(f"Saved CRMSE file: {output_path}")

Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54/TI_E05.npy using observational data E05_OBS
Saved CRMSE file: /glade/work/mhafsah/Analysis_Proj_1/errors_crmse_ti/era5_myj_vert54/CRMSE_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54/TI_E06.npy using observational data E06_OBS
Saved CRMSE file: /glade/work/mhafsah/Analysis_Proj_1/errors_crmse_ti/era5_myj_vert54/CRMSE_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80/TI_E05.npy using observational data E05_OBS
Saved CRMSE file: /glade/work/mhafsah/Analysis_Proj_1/errors_crmse_ti/era5_myj_vert80/CRMSE_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80/TI_E06.npy using observational data E06_OBS
Saved CRMSE file: /glade/work/mhafsah/Analysis_Proj_1/errors_crmse_ti/era5_myj_vert80/CRMSE_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert93/TI_E05.npy using observational data E05_OBS
Saved CRMSE file: 

--------------- Alternative code for cRMSE------------
# Function to compute centered RMSE
def centered_rmse(predicted, observed):
    # Remove the mean from both predicted and observed
    predicted_centered = predicted - np.mean(predicted)
    observed_centered = observed - np.mean(observed)
    
    # Calculate the RMSE between the centered values
    return np.sqrt(mean_squared_error(observed_centered, predicted_centered))

# Loop through WRF files
for f in wrf_files:
    ws_npy = np.load(f)

    # Check if the file corresponds to E05 or E06 and use the appropriate observational data
    if 'E05' in f:
        obs_data = WS_E05_OBS_interpolated  # Use E05 observational data
        obs_label = 'E05_OBS'
    elif 'E06' in f:
        obs_data = WS_E06_OBS_interpolated  # Use E06 observational data
        obs_label = 'E06_OBS'
    else:
        print(f"Skipping file {f}: Not matching E05 or E06.")
        continue  # Skip files that do not match E05 or E06

    # Initialize cRMSE array for 10 height levels
    cRMSE_WRF = np.zeros((obs_data.shape[0], 10))

    # Compute centered RMSE for each of the 10 height levels
    for i in range(10):
        cRMSE_WRF[:, i] = centered_rmse(ws_npy[:, i], obs_data[:, i])  # Use the correct obs_data

    # Replace "WRF" with "crmse" in the filename and save the result
    filename_crmse = os.path.basename(f).replace("WRF", "crmse")
    output_path = os.path.join(crmse_dir, filename_crmse)
    np.save(output_path, cRMSE_WRF)

    print(f"Saved CRMSE file: {output_path}")

-------------------Correlation Coefficient R2--------------------

In [18]:
import os
import glob
import numpy as np
import pandas as pd  # for interpolation

# Root where WS_all_runs is stored, change as necessary
# wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_all_runs"
wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_d02"

# Grab all WS_E05.npy and WS_E06.npy under each run directory
wrf_files = glob.glob(os.path.join(wrf_root, "*", "TI_E0*.npy"))

# Directory to save R² results (sibling of WS_all_runs)
r2_dir = os.path.join(os.path.dirname(wrf_root), "errors_r2_ti")
os.makedirs(r2_dir, exist_ok=True)

# Interpolate missing values in both E05 and E06 observational data
TI_E05_OBS_interpolated = (
    pd.DataFrame(TI_E05_OBS)
    .interpolate(method="linear", axis=0, limit_direction="both")
    .to_numpy()
)
TI_E06_OBS_interpolated = (
    pd.DataFrame(TI_E06_OBS)
    .interpolate(method="linear", axis=0, limit_direction="both")
    .to_numpy()
)

for f in sorted(wrf_files):
    ti_npy = np.load(f)

    base = os.path.basename(f)
    if "E05" in base:
        obs_data = TI_E05_OBS_interpolated
        obs_label = "E05_OBS"
    elif "E06" in base:
        obs_data = TI_E06_OBS_interpolated
        obs_label = "E06_OBS"
    else:
        print(f"Skipping file {f}: not matching E05 or E06.")
        continue

    # Sanity check on shapes
    if ti_npy.shape != obs_data.shape:
        print(f"Shape mismatch for {f}: WRF {ti_npy.shape}, OBS {obs_data.shape}")
        continue

    print(f"Processing {f} using observational data {obs_label}")

    # R² per height level → 10 numbers
    R2_TI = np.zeros(10)
    for i in range(10):
        r = np.corrcoef(ti_npy[:, i], obs_data[:, i])[0, 1]
        R2_TI[i] = r**2

    # Build a clean R² filename
    # e.g. WS_E05.npy → R2_WS_E05.npy
    filename_r2 = base.replace("TI_", "R2_TI_")

    run_name = os.path.basename(os.path.dirname(f))
    run_r2_dir = os.path.join(r2_dir, run_name)
    os.makedirs(run_r2_dir, exist_ok=True)

    output_path = os.path.join(run_r2_dir, filename_r2)
    np.save(output_path, R2_TI)

    print(f"Saved R² file: {output_path}")

Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54/TI_E05.npy using observational data E05_OBS
Saved R² file: /glade/work/mhafsah/Analysis_Proj_1/errors_r2_ti/era5_myj_vert54/R2_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54/TI_E06.npy using observational data E06_OBS
Saved R² file: /glade/work/mhafsah/Analysis_Proj_1/errors_r2_ti/era5_myj_vert54/R2_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80/TI_E05.npy using observational data E05_OBS
Saved R² file: /glade/work/mhafsah/Analysis_Proj_1/errors_r2_ti/era5_myj_vert80/R2_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80/TI_E06.npy using observational data E06_OBS
Saved R² file: /glade/work/mhafsah/Analysis_Proj_1/errors_r2_ti/era5_myj_vert80/R2_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert93/TI_E05.npy using observational data E05_OBS
Saved R² file: /glade/work/mhafsah/Analysis_Proj_1/err

-------------------- Earth Mover's Distance ------------------

In [20]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance

# Root where WS_all_runs is stored, change as necessary
# wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_all_runs"
wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_d02"

# Grab all WS_E05.npy and WS_E06.npy under each run directory
wrf_files = glob.glob(os.path.join(wrf_root, "*", "TI_E0*.npy"))

# Directory to save EMD results (sibling of WS_all_runs)
emd_dir = os.path.join(os.path.dirname(wrf_root), "errors_emd_ti")
os.makedirs(emd_dir, exist_ok=True)

# Interpolate missing values in both E05 and E06 observational data
TI_E05_OBS_interpolated = (
    pd.DataFrame(TI_E05_OBS)
    .interpolate(method="linear", axis=0, limit_direction="both")
    .to_numpy()
)
TI_E06_OBS_interpolated = (
    pd.DataFrame(TI_E06_OBS)
    .interpolate(method="linear", axis=0, limit_direction="both")
    .to_numpy()
)

for f in sorted(wrf_files):
    ti_npy = np.load(f)

    base = os.path.basename(f)
    if "E05" in base:
        obs_data = TI_E05_OBS_interpolated
        obs_label = "E05_OBS"
    elif "E06" in base:
        obs_data = TI_E06_OBS_interpolated
        obs_label = "E06_OBS"
    else:
        print(f"Skipping file {f}: not matching E05 or E06.")
        continue

    # Sanity check on shapes
    if ti_npy.shape != obs_data.shape:
        print(f"Shape mismatch for {f}: WRF {ti_npy.shape}, OBS {obs_data.shape}")
        continue

    print(f"Processing {f} using observational data {obs_label}")

    # EMD per height level → 10 numbers
    EMD_TI = np.zeros(10)
    for i in range(10):
        EMD_TI[i] = wasserstein_distance(ti_npy[:, i], obs_data[:, i])

    # Build a clean EMD filename
    # e.g. WS_E05.npy → EMD_WS_E05.npy
    filename_emd = base.replace("TI_", "EMD_TI_")

    run_name = os.path.basename(os.path.dirname(f))
    run_emd_dir = os.path.join(emd_dir, run_name)
    os.makedirs(run_emd_dir, exist_ok=True)

    output_path = os.path.join(run_emd_dir, filename_emd)
    np.save(output_path, EMD_TI)

    print(f"Saved EMD file: {output_path}")

Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54/TI_E05.npy using observational data E05_OBS
Saved EMD file: /glade/work/mhafsah/Analysis_Proj_1/errors_emd_ti/era5_myj_vert54/EMD_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54/TI_E06.npy using observational data E06_OBS
Saved EMD file: /glade/work/mhafsah/Analysis_Proj_1/errors_emd_ti/era5_myj_vert54/EMD_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80/TI_E05.npy using observational data E05_OBS
Saved EMD file: /glade/work/mhafsah/Analysis_Proj_1/errors_emd_ti/era5_myj_vert80/EMD_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80/TI_E06.npy using observational data E06_OBS
Saved EMD file: /glade/work/mhafsah/Analysis_Proj_1/errors_emd_ti/era5_myj_vert80/EMD_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert93/TI_E05.npy using observational data E05_OBS
Saved EMD file: /glade/work/mhafsah/Analys

----------------- Standard Deviation ---------------

In [65]:
# Root where WS_all_runs is stored, change as necessary. also change WS/WD/TI when required
# wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/WD_all_runs"
wrf_root = "/glade/work/mhafsah/Analysis_Proj_1/WD_d02"

# Grab all WS_E05.npy and WS_E06.npy under each run directory
wrf_files = glob.glob(os.path.join(wrf_root, "*", "WD_E0*.npy"))

# Directory to save standard deviation results (sibling of WS_all_runs)
sd_dir = os.path.join(os.path.dirname(wrf_root), "errors_sd_wd")
os.makedirs(sd_dir, exist_ok=True)

for f in sorted(wrf_files):
    wd_npy = np.load(f)

    base = os.path.basename(f)
    if "E05" in base:
        obs_label = "E05_OBS"
    elif "E06" in base:
        obs_label = "E06_OBS"
    else:
        print(f"Skipping file {f}: not matching E05 or E06.")
        continue

    print(f"Processing {f} ({obs_label})")

    # Standard deviation per height level → 10 numbers
    SD_WD = np.zeros(10)
    for i in range(10):
        SD_WD[i] = np.std(wd_npy[:, i])

    # Build a clean SD filename
    # e.g. WS_E05.npy → SD_WS_E05.npy
    filename_sd = base.replace("WD_", "SD_WD_")

    run_name = os.path.basename(os.path.dirname(f))
    run_sd_dir = os.path.join(sd_dir, run_name)
    os.makedirs(run_sd_dir, exist_ok=True)

    output_path = os.path.join(run_sd_dir, filename_sd)
    np.save(output_path, SD_WD)

    print(f"Saved standard deviation file: {output_path}")

Processing /glade/work/mhafsah/Analysis_Proj_1/WD_d02/era5_myj_vert54/WD_E05.npy (E05_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_wd/era5_myj_vert54/SD_WD_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/WD_d02/era5_myj_vert54/WD_E06.npy (E06_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_wd/era5_myj_vert54/SD_WD_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/WD_d02/era5_myj_vert80/WD_E05.npy (E05_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_wd/era5_myj_vert80/SD_WD_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/WD_d02/era5_myj_vert80/WD_E06.npy (E06_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_wd/era5_myj_vert80/SD_WD_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/WD_d02/era5_myj_vert93/WD_E05.npy (E05_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_wd/era5_myj_vert93/SD_WD_E05

In [22]:
import os
import glob
import numpy as np

# Root where TI is stored (change as necessary)
ti_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_all_runs"
# ti_root = "/glade/work/mhafsah/Analysis_Proj_1/TI_d02"

# Grab all TI_E05*.npy and TI_E06*.npy under each run directory
ti_files = glob.glob(os.path.join(ti_root, "*", "TI_E0*.npy"))

# Directory to save standard deviation results (sibling of TI_d02)
sd_dir = os.path.join(os.path.dirname(ti_root), "errors_sd_ti")
os.makedirs(sd_dir, exist_ok=True)

for f in sorted(ti_files):
    ti_npy = np.load(f)

    base = os.path.basename(f)
    if "E05" in base:
        obs_label = "E05_OBS"
    elif "E06" in base:
        obs_label = "E06_OBS"
    else:
        print(f"Skipping file {f}: not matching E05 or E06.")
        continue

    print(f"Processing {f} ({obs_label})")

    # Standard deviation per height level → 10 numbers
    SD_TI = np.zeros(10)
    for i in range(10):
        SD_TI[i] = np.std(ti_npy[:, i])

    # Build a clean SD filename
    # e.g. TI_E05.npy → SD_TI_E05.npy
    filename_sd = base.replace("TI_", "SD_TI_")

    run_name = os.path.basename(os.path.dirname(f))
    run_sd_dir = os.path.join(sd_dir, run_name)
    os.makedirs(run_sd_dir, exist_ok=True)

    output_path = os.path.join(run_sd_dir, filename_sd)
    np.save(output_path, SD_TI)

    print(f"Saved standard deviation file: {output_path}")

Processing /glade/work/mhafsah/Analysis_Proj_1/TI_all_runs/era5_3dpbl/TI_E05.npy (E05_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_ti/era5_3dpbl/SD_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_all_runs/era5_3dpbl/TI_E06.npy (E06_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_ti/era5_3dpbl/SD_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_all_runs/era5_3dpbl_vert54/TI_E05.npy (E05_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_ti/era5_3dpbl_vert54/SD_TI_E05.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_all_runs/era5_3dpbl_vert54/TI_E06.npy (E06_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_ti/era5_3dpbl_vert54/SD_TI_E06.npy
Processing /glade/work/mhafsah/Analysis_Proj_1/TI_all_runs/era5_3dpbl_vert80/TI_E05.npy (E05_OBS)
Saved standard deviation file: /glade/work/mhafsah/Analysis_Proj_1/errors_sd_ti/era5_3dpbl

In [25]:
import numpy as np
import os

# Path to the directory where R² files are saved
dir = "errors_bias_ws"

# List all files in the r2_dir directory
files = os.listdir(dir)

# Choose one file to read back (for example, the first file)
file_path = os.path.join(dir, files[7])

# Load the R² values from the file
error_WRF_loaded = np.load(file_path)

# Print the loaded R² values
print(f"Loaded error metrics values from {file_path}:")
print(error_WRF_loaded)

Loaded error metrics values from errors_bias_ws/f2_s0_icbc0_pbl5_WS_E06_bias.npy:
[[ 9.10698114e-01 -1.20622063e-01 -1.02625125e+00 -8.66888358e-01
  -8.65128830e-01 -8.06676880e-01 -9.66021054e-01 -1.67703304e+00
  -2.29673151e+00 -2.34368470e+00]
 [ 8.76203169e-01 -2.75206526e-01 -9.10887598e-01 -1.34500692e+00
  -8.57876003e-01 -7.43310213e-01 -1.45589928e+00 -2.43236517e+00
  -2.70827106e+00 -2.54999771e+00]
 [ 1.00304719e+00 -1.40158745e-01 -7.49954530e-01 -1.22129847e+00
  -5.99090266e-01 -9.40109250e-01 -1.83145961e+00 -2.47218551e+00
  -2.71511521e+00 -2.50569751e+00]
 [ 4.01058398e-01 -3.49741084e-01 -7.91697174e-01 -1.11002780e+00
  -1.16271422e+00 -1.58250289e+00 -2.51481537e+00 -3.24019703e+00
  -3.15576490e+00 -3.00429819e+00]
 [ 3.96449289e-01 -3.49612639e-01 -8.06140149e-01 -1.28692812e+00
  -1.29899777e+00 -1.88277983e+00 -2.72717575e+00 -3.22561173e+00
  -3.20900605e+00 -2.87444831e+00]
 [ 3.78570587e-02 -7.35156358e-01 -1.01306978e+00 -1.12044282e+00
  -1.02824544e+00

In [20]:
import numpy as np
import os

# Path to the directory where R² files are saved
dir = "errors_bias_ws"

# List all files in the r2_dir directory
files = os.listdir(dir)

# Choose one file to read back (for example, the first file)
file_path = os.path.join(dir, files[7])

# Load the R² values from the file
error_WRF_loaded = np.load(file_path)

# Print the loaded R² values
print(f"Loaded error metrics values from {file_path}:")
print(error_WRF_loaded)

Loaded error metrics values from errors_bias_ws/f2_s0_icbc0_pbl5_WS_E06_bias.npy:
[[ 1.86998114e-01 -5.84222063e-01 -1.22195125e+00 -1.52648836e+00
  -2.15792883e+00 -1.85427688e+00 -2.13972105e+00 -2.67243304e+00
              nan             nan]
 [ 6.74103169e-01 -2.60506526e-01 -7.75687598e-01 -1.31760692e+00
  -2.27477600e+00 -2.60651021e+00 -2.96479928e+00 -3.75236517e+00
              nan             nan]
 [ 8.84247186e-01 -1.21658745e-01 -8.45354530e-01 -1.44329847e+00
  -1.68639027e+00 -1.72370925e+00 -2.34555961e+00             nan
              nan             nan]
 [ 5.64558398e-01 -3.34141084e-01 -1.26059717e+00 -1.92432780e+00
  -1.65141422e+00 -1.55830289e+00 -2.08411537e+00 -2.65589703e+00
  -3.06466490e+00 -2.52319819e+00]
 [ 7.45349289e-01 -3.73512639e-01 -1.26414015e+00 -1.97622812e+00
  -1.37179777e+00 -1.76257983e+00 -2.49737575e+00 -3.14361173e+00
  -3.50650605e+00             nan]
 [ 1.23545706e+00  3.17043642e-01 -2.33669778e-01 -8.83242816e-01
  -1.34504544e+00

In [1]:
import numpy as np
import os

# Path to the directory where R² files are saved
dir = "errors_r2"

# List all files in the r2_dir directory
files = os.listdir(dir)

# Choose one file to read back (for example, the first file)
file_path = os.path.join(dir, files[0])

# Load the R² values from the file
error_WRF_loaded = np.load(file_path)

# Print the loaded R² values
print(f"Loaded error metrics values from {file_path}:")
print(error_WRF_loaded)

Loaded error metrics values from errors_r2/f2_s0_icbc0_pbl8_WS_E05_r2.npy:
[0.71677646 0.66613322 0.5596277  0.70630168 0.83893727 0.84306142
 0.7772814  0.71315972 0.70263444 0.76261456]


In [16]:
####### Renaming folders with desired naming style

import os

root = "/glade/work/mhafsah/Analysis_Proj_1/TI_d02"

for name in sorted(os.listdir(root)):
    old_path = os.path.join(root, name)
    if not os.path.isdir(old_path):
        continue

    if not name.endswith("_d02"):
        # leave other folders alone
        continue

    new_name = name[:-4]  # drop the last 4 chars: "_d02"
    new_path = os.path.join(root, new_name)

    print(f"{old_path}  ->  {new_path}")
    os.rename(old_path, new_path)

/glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54_d02  ->  /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert54
/glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80_d02  ->  /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert80
/glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert93_d02  ->  /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_myj_vert93
/glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_mynn_vert54_d02  ->  /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_mynn_vert54
/glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_mynn_vert80_d02  ->  /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_mynn_vert80
/glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_mynn_vert93_d02  ->  /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_mynn_vert93
/glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_ysu_vert54_d02  ->  /glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_ysu_vert54
/glade/work/mhafsah/Analysis_Proj_1/TI_d02/era5_ysu_vert80_d02  ->  /glade/work/mhafsah/Analysis_Proj_1/T